Antes de iniciar o projeto de fato, primeiro fazemos a analise exploratoria para garatir a qualidade da fonte de dados.
Neste caso utilizamos o ML canvas para entender o negocio e o problema da operadora e utilizamos o EDA para analisar os dados e nos guiar durante o projeto

Blibliotecas: Pandas(manipulação de dados) e Seaborn/Matplotlib(graficos).


In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

sns.set_theme(style="whitegrid")
#ler  CSV que esta na pasta DATA/RAW 
df = pd.read_csv('../data/raw/Telco_Customer_Churn.csv')

#Exibir as 5 primeiras linhas do arquivo
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Agora vamos entender a quantidade de linhas e colunas existentes nessa fonte de dados com o df.shape[ ] 
[0] -> para linhas
[1] -> para Colunas 
E com o df.info() podemos ter as mias informações sobre a tipagem dos dados

In [27]:
print(f"Temos {df.shape[0]} linhas representando os clientes e {df.shape[1]} colunas, representando as nossas váriaveis.")
print (f"------------------------------------------------")
print (f"Aqui abaixo informações sobre nossas váriaveis")

df.info()

Temos 7043 linhas representando os clientes e 21 colunas, representando as nossas váriaveis.
------------------------------------------------
Aqui abaixo informações sobre nossas váriaveis
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  Strea

Podemos entender que temo 7043 linha e em nenhuma váriavel temos campos nulos, também podemos entender através do Dtype os tipos dos dados que estamos trabalhando. Com base na tabela e nas 5 primeiras linhas, entendo que algumas váriaveis STR devem virar binárias. Então vamos fazer o encoding desses campos. Para poder entender melhor os campos string e definir o que vamos fazer com eles, vamos selecionar os campos e fazer um mini relatório.

In [28]:
colunas_texto =  df.select_dtypes(include=['string']).columns

for col in colunas_texto:
    print(f"---Variavel: {col}---")
    print(f"Opções: {df[col].nunique()}")
    print(df[col].value_counts())
    print("-"*40)

---Variavel: customerID---
Opções: 7043
customerID
7590-VHVEG    1
5575-GNVDE    1
3668-QPYBK    1
7795-CFOCW    1
9237-HQITU    1
             ..
6840-RESVB    1
2234-XADUH    1
4801-JZAZL    1
8361-LTMKD    1
3186-AJIEK    1
Name: count, Length: 7043, dtype: int64
----------------------------------------
---Variavel: gender---
Opções: 2
gender
Male      3555
Female    3488
Name: count, dtype: int64
----------------------------------------
---Variavel: Partner---
Opções: 2
Partner
No     3641
Yes    3402
Name: count, dtype: int64
----------------------------------------
---Variavel: Dependents---
Opções: 2
Dependents
No     4933
Yes    2110
Name: count, dtype: int64
----------------------------------------
---Variavel: PhoneService---
Opções: 2
PhoneService
Yes    6361
No      682
Name: count, dtype: int64
----------------------------------------
---Variavel: MultipleLines---
Opções: 3
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count

Com isso entendo que devemosfazer a seguinte conversão
Campos para fazer encoding
Yes 1, No 0
Partner
Depedents
PhoneService
OnlineSecurity
DeviceProtection
TechSupport
StreamingTV
StreamingMovies
PaperlessBilling
Churn

Mudar para numérico
MonthlyCharges
TotalCharges

one hot ecoding
Multiplelines
InternetService
OnlineSecurity
OnlineBackup
DeviceProtection
TechSupport
StreamingTV
StreamingMovies
Contract
PaymentMethod

In [ ]:
#1ª Etapa converter campos strig que podem ser numéricos em numéricos e tratar os campos nulos 
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'],  errors='coerce')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'],  errors='coerce')

nulos_antes = df['MonthlyCharges'].isnull().sum()
print(f"Valores nulos encontrados em MonthlyCharges: {nulos_antes}")

nulos_antes = df['TotalCharges'].isnull().sum()
print(f"Valores nulos encontrados em TotalCharges: {nulos_antes}")

media_total_charges = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(media_total_charges)

print(f"Nulos tratados para total charges. novos nulos: {df['TotalCharges'].isnull().sum()}")

#Tratamento para transformar os campos em binários Yes = 1 No = 0

campos_binarios =  [
    'Partner', 'Dependents', 'PhoneService', 'OnlineSecurity' 
    ,'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
    'PaperlessBilling', 'Churn' ]

for col in campos_binarios:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df['gender'] = df['gender'].map({'Female': 1, 'Male':0})
#Tratamento one-hot Encoding 

colunas_multi = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
    'Contract', 'PaymentMethod'
]

df = pd.get_dummies(df, columns=colunas_multi, drop_first=True)

print(f'Agora temos o total de {df.shape[1]} colunas.')


Valores nulos encontrados em MonthlyCharges: 0
Valores nulos encontrados em TotalCharges: 11
Nulos tratados para total charges. novos nulos: 0
Agora temos o total de 27 colunas.


In [33]:
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,OnlineBackup_Yes,DeviceProtection_1.0,TechSupport_1.0,StreamingTV_1.0,StreamingMovies_1.0,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,Female,0,1,0,1,0,1,29.85,29.85,...,True,False,False,False,False,False,False,False,True,False
1,5575-GNVDE,Male,0,0,0,34,1,0,56.95,1889.50,...,False,True,False,False,False,True,False,False,False,True
2,3668-QPYBK,Male,0,0,0,2,1,1,53.85,108.15,...,True,False,False,False,False,False,False,False,False,True
3,7795-CFOCW,Male,0,0,0,45,0,0,42.30,1840.75,...,False,True,True,False,False,True,False,False,False,False
4,9237-HQITU,Female,0,0,0,2,1,1,70.70,151.65,...,False,False,False,False,False,False,False,False,True,False


In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 27 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   customerID                             7043 non-null   str    
 1   gender                                 7043 non-null   str    
 2   SeniorCitizen                          7043 non-null   int64  
 3   Partner                                7043 non-null   int64  
 4   Dependents                             7043 non-null   int64  
 5   tenure                                 7043 non-null   int64  
 6   PhoneService                           7043 non-null   int64  
 7   PaperlessBilling                       7043 non-null   int64  
 8   MonthlyCharges                         7043 non-null   float64
 9   TotalCharges                           7043 non-null   float64
 10  Churn                                  7043 non-null   int64  
 11  MultipleLines_N